# Approach 3.3 — GPT-2 Decoder with Embedding Prefix

## Why GPT-2?

The LSTM baseline (03_1) fails on out-of-vocabulary words — it cannot generate "Carvallo", "SNCF", or "hacking" because they never appeared in the training sentences.

DistilGPT-2 solves this with a **BPE tokenizer** (50,257 subword tokens): every word can be represented as a sequence of subwords, so OOV is impossible by construction.

The pretrained LM weights also provide strong language priors that help generalize from only 35 K training pairs.

## Architecture

A single linear projection maps the sentence embedding (768-d) to DistilGPT-2's hidden dimension.  
The projected vector is prepended as a **prefix token** before the actual token sequence.  
The model learns to generate the sentence conditioned on this prefix.

This is a simplified form of prefix-tuning: only a linear layer and the full GPT-2 model are fine-tuned jointly.

In [ ]:
%matplotlib inline

import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

import copy
import faiss
import faulthandler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

TRAIN_DATA_FILE   = 'data/sentences_train_text_db.parquet'
TRAIN_INDEX_FILE  = 'data/sentences_train_vector_db.index'
TARGET_DATA_FILE  = 'data/sentences_target_text_db.parquet'
TARGET_INDEX_FILE = 'data/sentences_target_vector_db.index'

GPT2_NAME    = 'distilgpt2'
MAX_LEN      = 48
BATCH_SIZE   = 16
EPOCHS       = 40
PATIENCE     = 8
LR           = 5e-4
D_MODEL      = 192
N_HEAD       = 6
N_LAYERS     = 3
DROPOUT      = 0.1
VAL_SIZE     = 0.2
RANDOM_SEED  = 42
SAMPLE_EVERY = 10
VAL_EVERY    = 2
USE_MPS      = True
NEG_MASK_VAL = -1e4

faulthandler.enable(all_threads=True)
torch.manual_seed(RANDOM_SEED)
torch.set_num_threads(1)
DEVICE = torch.device('mps' if USE_MPS and torch.backends.mps.is_available() else 'cpu')
print(f'device : {DEVICE}')

In [ ]:
df          = pd.read_parquet(TRAIN_DATA_FILE)
faiss_index = faiss.read_index(TRAIN_INDEX_FILE)
all_emb     = np.zeros((faiss_index.ntotal, faiss_index.d), dtype=np.float32)
faiss_index.reconstruct_n(0, faiss_index.ntotal, all_emb)

SENTENCE_EMB_DIM = faiss_index.d
embeddings       = all_emb[df['id'].values]
sentences        = df['text'].tolist()
del all_emb

print(f'sentences: {len(sentences):,}')
print(f'emb shape: {embeddings.shape}')

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained(GPT2_NAME)
tokenizer.pad_token = tokenizer.eos_token

PAD_ID = tokenizer.eos_token_id

print(f'vocab size : {tokenizer.vocab_size:,}')
print(f'PAD / EOS: {PAD_ID}')

# Quick OOV check on a target sentence — compare with LSTM baseline
sample_target = "I'm Nicolas Carvallo. I like hacking SNCF and booking a whole train booth just to be alone."
tokens = tokenizer.tokenize(sample_target)
print(f'\nTarget sentence BPE tokens ({len(tokens)}) : {tokens}')

In [ ]:
import gc

class EmbeddingTextDataset(Dataset):
    def __init__(self, embeddings: np.ndarray, sentences: list[str]):
        self.embeddings = torch.from_numpy(np.ascontiguousarray(embeddings))

        # Pre-tokenize once to avoid re-running GPT2 tokenizer at every sample/epoch.
        enc = tokenizer(
            sentences,
            max_length=MAX_LEN,
            truncation=True,
            padding='max_length',
            return_tensors='np',
        )
        ids = torch.from_numpy(enc['input_ids']).long()
        mask = torch.from_numpy(enc['attention_mask']).long()

        dec_input = torch.full_like(ids, PAD_ID)
        dec_input[:, 1:] = ids[:, :-1]

        dec_mask = torch.zeros_like(mask)
        dec_mask[:, 0] = 1
        dec_mask[:, 1:] = mask[:, :-1]

        labels = ids.clone()
        labels[mask == 0] = -100

        self.dec_input = dec_input
        self.dec_mask = dec_mask
        self.labels = labels

    def __len__(self) -> int:
        return len(self.embeddings)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        return self.embeddings[idx], self.dec_input[idx], self.dec_mask[idx], self.labels[idx]


emb_train, emb_val, sent_train, sent_val = train_test_split(
    embeddings, sentences, test_size=VAL_SIZE, random_state=RANDOM_SEED
)

train_loader = DataLoader(EmbeddingTextDataset(emb_train, sent_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(EmbeddingTextDataset(emb_val,   sent_val),   batch_size=BATCH_SIZE, shuffle=False)

print(f'train: {len(sent_train):,} | val: {len(sent_val):,}')

del faiss_index, embeddings, df
gc.collect()

In [ ]:
import gc


# ── custom building blocks — only matmul/softmax, no SDPA ───────────────────

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, nhead: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % nhead == 0
        self.nhead  = nhead
        self.d_head = d_model // nhead
        self.scale  = self.d_head ** -0.5
        self.wq = nn.Linear(d_model, d_model, bias=False)
        self.wk = nn.Linear(d_model, d_model, bias=False)
        self.wv = nn.Linear(d_model, d_model, bias=False)
        self.wo = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        attn_mask: torch.Tensor | None = None,
        key_padding_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        B, Tq, _ = q.shape
        Tk = k.shape[1]
        H, Dh = self.nhead, self.d_head

        q = self.wq(q).view(B, Tq, H, Dh).transpose(1, 2)
        k = self.wk(k).view(B, Tk, H, Dh).transpose(1, 2)
        v = self.wv(v).view(B, Tk, H, Dh).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if attn_mask is not None:
            scores = scores + attn_mask
        if key_padding_mask is not None:
            scores = scores.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2), NEG_MASK_VAL)

        weights = self.drop(F.softmax(scores.float(), dim=-1).to(scores.dtype))
        out = torch.matmul(weights, v).transpose(1, 2).contiguous().view(B, Tq, H * Dh)
        return self.wo(out)


class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, nhead: int, dropout: float = 0.0):
        super().__init__()
        self.sa    = MultiHeadAttention(d_model, nhead, dropout)
        self.xa    = MultiHeadAttention(d_model, nhead, dropout)
        self.ff1   = nn.Linear(d_model, d_model * 4)
        self.ff2   = nn.Linear(d_model * 4, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        mem: torch.Tensor,
        tgt_mask: torch.Tensor | None = None,
        tgt_key_padding_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        nx = self.norm1(x)
        x  = x + self.drop(self.sa(nx, nx, nx, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask))
        nx = self.norm2(x)
        x  = x + self.drop(self.xa(nx, mem, mem))
        x  = x + self.drop(self.ff2(self.drop(F.gelu(self.ff1(self.norm3(x))))))
        return x


# ── step 1: test MultiHeadAttention on CPU ───────────────────────────────────
print("[1/5] MultiHeadAttention (matmul/softmax only) on CPU...")
_a = MultiHeadAttention(64, 4)
_o = _a(torch.randn(2, 5, 64), torch.randn(2, 1, 64), torch.randn(2, 1, 64))
print(f"OK — {_o.shape}")
del _a, _o
gc.collect()

# ── step 2: test DecoderBlock on CPU ─────────────────────────────────────────
print("[2/5] DecoderBlock on CPU...")
_b = DecoderBlock(64, 4)
_o = _b(torch.randn(2, 5, 64), torch.randn(2, 1, 64))
print(f"OK — {_o.shape}")
del _b, _o
gc.collect()

# ── step 3: define full model ─────────────────────────────────────────────────
print("[3/5] defining EmbeddingConditionedDecoder...")


class EmbeddingConditionedDecoder(nn.Module):
    def __init__(
        self,
        sentence_emb_dim: int,
        vocab_size: int,
        d_model: int = D_MODEL,
        nhead: int = N_HEAD,
        num_layers: int = N_LAYERS,
        dropout: float = DROPOUT,
    ):
        super().__init__()
        self.proj    = nn.Linear(sentence_emb_dim, d_model)
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_emb = nn.Embedding(MAX_LEN + 1, d_model)
        self.blocks  = nn.ModuleList([DecoderBlock(d_model, nhead, dropout) for _ in range(num_layers)])
        self.norm    = nn.LayerNorm(d_model)
        # Independent head weights — weight tying with tok_emb causes MPS kernel crash on .to(device)
        self.head    = nn.Linear(d_model, vocab_size, bias=False)

    def forward(
        self,
        sentence_emb:   torch.Tensor,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        labels:         torch.Tensor | None = None,
    ) -> torch.Tensor:
        B, T  = input_ids.shape
        pos   = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x     = self.tok_emb(input_ids) + self.pos_emb(pos)
        mem   = self.proj(sentence_emb).unsqueeze(1)
        cmask = torch.triu(torch.full((T, T), NEG_MASK_VAL, device=x.device, dtype=x.dtype), diagonal=1)
        kpm   = (attention_mask == 0) if attention_mask is not None else None
        for block in self.blocks:
            x = block(x, mem, tgt_mask=cmask, tgt_key_padding_mask=kpm)
        logits = self.head(self.norm(x))
        if labels is not None:
            return F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100,
            )
        return logits


print("OK")

# ── step 4: instantiate on CPU ────────────────────────────────────────────────
print("[4/5] instantiating on CPU...")
model = EmbeddingConditionedDecoder(SENTENCE_EMB_DIM, tokenizer.vocab_size)
print(f"OK — {sum(p.numel() for p in model.parameters()):,} params")

# ── step 5: move to device ────────────────────────────────────────────────────
print(f"[5/5] moving to {DEVICE}...")
model = model.to(DEVICE)
print(f"OK — on {DEVICE}")

# smoke test on selected device before starting the full training loop
_emb  = torch.randn(2, SENTENCE_EMB_DIM, device=DEVICE)
_ids  = torch.full((2, MAX_LEN), PAD_ID, dtype=torch.long, device=DEVICE)
_mask = torch.ones((2, MAX_LEN), dtype=torch.long, device=DEVICE)
_lbl  = _ids.clone()
_loss = model(_emb, _ids, attention_mask=_mask, labels=_lbl)
print(f"device smoke test OK — loss={_loss.detach().item():.4f}")
del _emb, _ids, _mask, _lbl, _loss
if DEVICE.type == 'mps':
    torch.mps.empty_cache()
gc.collect()

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
print("optimizer ready")

In [ ]:
def greedy_decode(emb: torch.Tensor) -> str:
    """Generate a sentence token by token from a single sentence embedding."""
    model.eval()
    with torch.no_grad():
        mem     = model.proj(emb).unsqueeze(1)
        cur_ids = torch.full((1, 1), PAD_ID, dtype=torch.long, device=DEVICE)
        result  = []
        for _ in range(MAX_LEN):
            T     = cur_ids.shape[1]
            pos   = torch.arange(T, device=DEVICE).unsqueeze(0)
            x     = model.tok_emb(cur_ids) + model.pos_emb(pos)
            cmask = torch.triu(torch.full((T, T), NEG_MASK_VAL, device=DEVICE, dtype=x.dtype), diagonal=1)
            for block in model.blocks:
                x = block(x, mem, tgt_mask=cmask)
            x   = model.norm(x)
            nxt = model.head(x[:, -1]).argmax(dim=-1).item()
            if nxt == tokenizer.eos_token_id:
                break
            result.append(nxt)
            cur_ids = torch.cat([cur_ids, torch.tensor([[nxt]], device=DEVICE)], dim=1)
    return tokenizer.decode(result, skip_special_tokens=True)


sample_emb = torch.tensor(emb_val[0], dtype=torch.float32).unsqueeze(0).to(DEVICE)
sample_ref = sent_val[0]

train_losses:  list[float] = []
val_losses:    list[float] = []
best_val_loss  = float('inf')
patience_count = 0
best_state     = None
last_val       = float('nan')

pbar = tqdm(range(1, EPOCHS + 1), desc='training', unit='epoch')
for epoch in pbar:
    model.train()
    total_train = 0.0
    for emb_batch, ids_batch, mask_batch, lbl_batch in train_loader:
        emb_batch  = emb_batch.to(DEVICE)
        ids_batch  = ids_batch.to(DEVICE)
        mask_batch = mask_batch.to(DEVICE)
        lbl_batch  = lbl_batch.to(DEVICE)
        loss       = model(emb_batch, ids_batch, attention_mask=mask_batch, labels=lbl_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_train += loss.item()

    avg_train = total_train / len(train_loader)
    train_losses.append(avg_train)

    if epoch == 1 or epoch % VAL_EVERY == 0:
        model.eval()
        total_val = 0.0
        with torch.no_grad():
            for emb_batch, ids_batch, mask_batch, lbl_batch in val_loader:
                emb_batch  = emb_batch.to(DEVICE)
                ids_batch  = ids_batch.to(DEVICE)
                mask_batch = mask_batch.to(DEVICE)
                lbl_batch  = lbl_batch.to(DEVICE)
                total_val += model(emb_batch, ids_batch, attention_mask=mask_batch, labels=lbl_batch).item()

        avg_val = total_val / len(val_loader)
        last_val = avg_val
        val_losses.append(avg_val)

        if avg_val < best_val_loss:
            best_val_loss  = avg_val
            patience_count = 0
            best_state     = copy.deepcopy(model.state_dict())
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                tqdm.write(f'early stop at epoch {epoch}  —  best val {best_val_loss:.4f}')
                break
    else:
        val_losses.append(float('nan'))

    pbar.set_postfix(train=f'{avg_train:.4f}', val=f'{last_val:.4f}', patience=patience_count)

    if epoch % SAMPLE_EVERY == 0 or epoch == 1:
        preview = greedy_decode(sample_emb)
        tqdm.write(f'epoch {epoch:3d}  |  "{preview}"')
        tqdm.write(f'ref: "{sample_ref[:80]}"')

model.load_state_dict(best_state)
print(f'restored best weights (val {best_val_loss:.4f})')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
epochs_range = range(1, len(train_losses) + 1)
ax.plot(epochs_range, train_losses, label='train')
ax.plot(epochs_range, val_losses,   label='val', linestyle='--')
ax.set_xlabel('epoch')
ax.set_ylabel('cross-entropy loss')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
target_meta_df = pd.read_parquet(TARGET_DATA_FILE)
target_fi      = faiss.read_index(TARGET_INDEX_FILE)
target_all_emb = np.zeros((target_fi.ntotal, target_fi.d), dtype=np.float32)
target_fi.reconstruct_n(0, target_fi.ntotal, target_all_emb)

print(f'{len(target_meta_df)} target sentences loaded')

In [ ]:
for _, row in target_meta_df.iterrows():
    target_emb = torch.tensor(target_all_emb[row['id']], dtype=torch.float32).unsqueeze(0).to(DEVICE)
    generated  = greedy_decode(target_emb)
    truth      = row['text']

    gen_toks    = generated.split()
    truth_toks  = truth.split()
    match_count = sum(g == t for g, t in zip(gen_toks, truth_toks))
    max_len_cmp = max(len(gen_toks), len(truth_toks))
    token_acc   = match_count / max_len_cmp if max_len_cmp > 0 else 0.0

    print(f"{row['target_id']}")
    print(f'generated: {generated}')
    print(f'ground truth: {truth}')
    print(f'token acc: {token_acc:.0%} | exact: {"yes" if generated == truth else "no"}')
    print()